In [7]:
!pip install -q transformers evaluate rouge-score sacrebleu
!pip install -q torch pandas tqdm scikit-learn sentence-transformers


In [8]:
import json, torch, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from tqdm import tqdm
import evaluate
from rouge_score import rouge_scorer
from collections import defaultdict
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [9]:
with open("/content/BioASQ12b.json") as f:
    bioasq_data = json.load(f)

with open("/content/pubMed.json") as f:
    pubmedqa_data = json.load(f)


In [10]:
def preprocess_bioasq(data):
    rows = []
    for q in data["questions"]:
        if q.get("ideal_answer") and q.get("snippets"):
            rows.append({
                "question": q["body"],
                "context": " ".join(s["text"] for s in q["snippets"]),
                "answer": q["ideal_answer"][0]
            })
    return pd.DataFrame(rows)

def preprocess_pubmedqa(data):
    rows = []
    for _, v in data.items():
        if v.get("LONG_ANSWER"):
            rows.append({
                "question": v["QUESTION"],
                "context": " ".join(v["CONTEXTS"]),
                "answer": v["LONG_ANSWER"]
            })
    return pd.DataFrame(rows)

def split_df(df):
    train, temp = train_test_split(df, test_size=0.3, random_state=42)
    val, test = train_test_split(temp, test_size=0.5, random_state=42)
    return train, val, test


In [11]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**Preprocessing Agent**

In [12]:
def preprocessing_agent(question, context):
    words = context.split()
    if len(words) < 120:
        return context

    sentences = context.split(". ")
    cleaned = []

    for s in sentences:
        overlap = sum(1 for w in question.split() if w.lower() in s.lower())
        if overlap > 0:
            cleaned.append(s)

    return ". ".join(cleaned[:6]) if cleaned else " ".join(sentences[:5])


**Retrieval Agent**

In [13]:
def retrieval_agent(question, context, top_k=5):
    chunks = context.split(". ")
    scored = []

    for ch in chunks:
        score = sum(1 for w in question.split() if w.lower() in ch.lower())
        scored.append((score, ch))

    scored.sort(reverse=True)
    return " ".join([c for _, c in scored[:top_k]])


In [14]:
def apply_prvg(df):
    final_contexts = []
    for _, r in tqdm(df.iterrows(), total=len(df)):
        clean = preprocessing_agent(r["question"], r["context"])
        final = retrieval_agent(r["question"], clean)
        final_contexts.append(final)

    df = df.copy()
    df["final_context"] = final_contexts
    return df


In [15]:
# BioASQ
bio_df = preprocess_bioasq(bioasq_data)
bio_tr, bio_va, bio_te = split_df(bio_df)

bio_tr = apply_prvg(bio_tr)
bio_va = apply_prvg(bio_va)
bio_te = apply_prvg(bio_te)

# PubMedQA
pm_df = preprocess_pubmedqa(pubmedqa_data)
pm_tr, pm_va, pm_te = split_df(pm_df)

pm_tr = apply_prvg(pm_tr)
pm_va = apply_prvg(pm_va)
pm_te = apply_prvg(pm_te)


100%|██████████| 150/150 [00:00<00:00, 5643.12it/s]


In [16]:
class QADataset(Dataset):
    def __init__(self, df, max_in=512, max_out=128):
        self.df = df.reset_index(drop=True)
        self.max_in = max_in
        self.max_out = max_out

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.loc[idx]

        x = tokenizer(
            r["question"] + " " + r["final_context"],
            max_length=self.max_in,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        y = tokenizer(
            r["answer"],
            max_length=self.max_out,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": x["input_ids"].squeeze(),
            "attention_mask": x["attention_mask"].squeeze(),
            "labels": y["input_ids"].squeeze(),
            "question": r["question"],
            "context": r["final_context"]
        }


**WhyMedQA model**

In [17]:
class WhyMedQA(nn.Module):
    def __init__(self):
        super().__init__()
        self.bart = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
        h = self.bart.config.d_model

        self.attn = nn.MultiheadAttention(h, 8, batch_first=True)
        self.fc = nn.Linear(h, h)
        self.act = nn.GELU()
        self.drop = nn.Dropout(0.1)
        self.norm = nn.LayerNorm(h)

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.bart.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=labels[:, :-1] if labels is not None else None,
            return_dict=True
        )

        h = out.last_hidden_state
        a, _ = self.attn(h, h, h)
        z = self.norm(self.drop(self.act(self.fc(a))) + h)
        logits = self.bart.lm_head(z)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
            loss = loss_fn(
                logits.view(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1)
            )

        return {"loss": loss, "logits": logits}


In [18]:
def train_model(train_loader, epochs=10):
    model = WhyMedQA().to(device)
    opt = AdamW(model.parameters(), lr=2e-5)

    sch = get_linear_schedule_with_warmup(
        opt, 0, len(train_loader) * epochs
    )

    for e in range(epochs):
        model.train()
        tot_loss, tot_tok = 0, 0

        for b in tqdm(train_loader, desc=f"Epoch {e+1}"):
            ids = b["input_ids"].to(device)
            msk = b["attention_mask"].to(device)
            lbl = b["labels"].to(device)

            out = model(ids, msk, lbl)
            out["loss"].backward()

            non_pad = (lbl[:,1:] != tokenizer.pad_token_id).sum().item()
            tot_loss += out["loss"].item() * non_pad
            tot_tok += non_pad

            opt.step(); sch.step(); opt.zero_grad()

        print(f"Epoch {e+1} | Avg Train Loss: {tot_loss/tot_tok:.4f}")

    return model


**AGENT 3 — Question-Type Aware Generation**

In [29]:
def detect_question_type(question):
    if isinstance(question, list):
        question = question[0]
    q = question.lower()
    if q.startswith(("what is", "which", "define", "who")):
        return "definition"
    if q.startswith(("is", "are", "does", "do", "can", "have", "has")):
        return "yesno"
    return "general"


def generation_agent_m1(model, batch):
    # unwrap question
    question = batch["question"][0] if isinstance(batch["question"], list) else batch["question"]
    qtype = detect_question_type(question)

    # tensors are already [1, seq_len]
    ids = batch["input_ids"].to(device)
    msk = batch["attention_mask"].to(device)

    gen_kwargs = {
        "max_length": 128,
        "num_beams": 4
    }

    if qtype == "definition":
        gen_kwargs["length_penalty"] = 0.6
        gen_kwargs["repetition_penalty"] = 1.1

    with torch.no_grad():
        gen = model.bart.generate(
            input_ids=ids,
            attention_mask=msk,
            **gen_kwargs
        )

    return tokenizer.decode(gen[0], skip_special_tokens=True)


**Faithfulness Agent**

In [33]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def semantic_faithfulness(answer, context):
    # unwrap context if it comes as a list (batch_size = 1)
    if isinstance(context, list):
        context = context[0]

    a = embedder.encode(answer, convert_to_tensor=True)
    c = embedder.encode(context, convert_to_tensor=True)

    # ensure both are 1D vectors
    a = a.squeeze()
    c = c.squeeze()

    return F.cosine_similarity(a, c, dim=0).item()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
bleu = evaluate.load("bleu")

def evaluate_model_prvg_m1(model, loader):
    model.eval()
    preds, refs, faith = [], [], []

    scorer = rouge_scorer.RougeScorer(
        ["rouge1","rouge2","rougeL"], use_stemmer=True
    )

    for b in tqdm(loader):
        ans = generation_agent_m1(model, b)
        ref = tokenizer.decode(b["labels"], skip_special_tokens=True)

        preds.append(ans)
        refs.append(ref)
        faith.append(semantic_faithfulness(ans, b["context"]))

    bleu_s = bleu.compute(predictions=preds, references=refs)

    rouge_out = defaultdict(list)
    for p,r in zip(preds,refs):
        s = scorer.score(r,p)
        for k in s:
            rouge_out[k].append(s[k].fmeasure)

    return {
        "BLEU-1": bleu_s["precisions"][0],
        "BLEU-2": bleu_s["precisions"][1],
        "ROUGE-1": np.mean(rouge_out["rouge1"]),
        "ROUGE-2": np.mean(rouge_out["rouge2"]),
        "ROUGE-L": np.mean(rouge_out["rougeL"]),
        "Faithfulness": np.mean(faith)
    }


In [35]:
bleu = evaluate.load("bleu")

def evaluate_prvg_m1(model, loader):
    model.eval()
    preds, refs, faith = [], [], []

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=True
    )

    for b in tqdm(loader):
        ans = generation_agent_m1(model, b)

        # unwrap reference string
        ref = tokenizer.batch_decode(
            b["labels"], skip_special_tokens=True
        )[0]

        preds.append(ans)
        refs.append(ref)

        faith.append(
            semantic_faithfulness(ans, b["context"])
        )

    bleu_s = bleu.compute(predictions=preds, references=refs)

    rouge_out = defaultdict(list)
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        for k in s:
            rouge_out[k].append(s[k].fmeasure)

    return {
        "BLEU-1": bleu_s["precisions"][0],
        "BLEU-2": bleu_s["precisions"][1],
        "ROUGE-1": np.mean(rouge_out["rouge1"]),
        "ROUGE-2": np.mean(rouge_out["rouge2"]),
        "ROUGE-L": np.mean(rouge_out["rougeL"]),
        "Faithfulness": np.mean(faith)
    }


In [ ]:
model = train_model(
    DataLoader(QADataset(bio_tr), batch_size=2, shuffle=True, num_workers=2, pin_memory=True),
    epochs=10
)




Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Epoch 1: 100%|██████████| 1767/1767 [06:14<00:00,  4.72it/s]


Epoch 1 | Avg Train Loss: 1.9292


Epoch 2:  57%|█████▋    | 1000/1767 [03:31<02:42,  4.71it/s]

In [34]:
bio_results_m1 = evaluate_prvg_m1(
    model,
    DataLoader(QADataset(bio_te), batch_size=1)
)
pd.DataFrame([bio_results_m1])


100%|██████████| 758/758 [10:43<00:00,  1.18it/s]


AttributeError: 'list' object has no attribute 'lower'

In [ ]:
pm_results_m1 = evaluate_prvg_m1(
    model,
    DataLoader(QADataset(pm_te), batch_size=1)
)

pd.DataFrame([pm_results_m1])
